In [ ]:
%pip install -U transformers datasets peft accelerate bitsandbytes

In [ ]:
%pip install flash-attn --no-build-isolation # for A100 when we use flash-attn

In [ ]:
!git clone https://github.com/DimitrisKu/Active-Reading--Pattern-Recognition.git

import os

# %cd /kaggle/working/Active-Reading--Pattern-Recognition # for kaggle environment
%cd /content/Active-Reading--Pattern-Recognition # for colab environment

os.getcwd()

In [ ]:
import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from datasets import load_dataset, concatenate_datasets
from itertools import chain
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ---- FORCE SINGLE GPU (important) ----
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# --- Config ---
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"


# --- For SimpleWiki paraphrase data ---
# ORIGINAL DATA PATH = "Datasets/simple_wiki_corpus.json" # Check this
# DATA_PATH = "generated_simplewiki/paraphrase_outputs" # Check this
# --------------------------------------

# --- For FinanceBench paraphrase data ---
ORIGINAL_DATA_PATH = "Datasets/finance_bench_corpus.json" # Check this
DATA_PATH = "paraphrase_outputs" # Check this
# --------------------------------------


MAX_SEQ_LENGTH = 1024
LEARNING_RATE = 2e-4

# --- Dataset ---
paraphrase_dataset = load_dataset("json", data_files=DATA_PATH, split="train")
original_dataset = load_dataset("json", data_files=ORIGINAL_DATA_PATH, split="train")

# Keep only 50% of the original dataset
original_dataset = original_dataset.shuffle(seed=42)
num_samples_to_keep = len(original_dataset) // 2
original_subset = original_dataset.select(range(num_samples_to_keep))

dataset = concatenate_datasets([paraphrase_dataset, original_subset])

# --- Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- 4-bit Quantization Config ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    # bnb_4bit_compute_dtype=torch.float16, # for T4-GPU
    bnb_4bit_compute_dtype=torch.bfloat16, # for A100
    bnb_4bit_use_double_quant=True,
)

# --- Load Model (QLoRA) ---
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="flash_attention_2", # for A100
)

# --- Prep for k-bit training ---
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
model.gradient_checkpointing_enable()

# --- LoRA ---
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# --- Tokenization ---
def tokenize_function(examples):
    return tokenizer(examples["text"])

def group_texts(examples):
    concatenated = {k: list(chain(*examples[k])) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])
    total_length = (total_length // MAX_SEQ_LENGTH) * MAX_SEQ_LENGTH

    result = {
        k: [t[i:i + MAX_SEQ_LENGTH] for i in range(0, total_length, MAX_SEQ_LENGTH)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

tokenized = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names,
    num_proc=2
)

lm_dataset = tokenized.map(
    group_texts,
    batched=True,
    num_proc=2
)

# --- Training Args ---
training_args = TrainingArguments(
    output_dir="./qlora_qwen4b",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=LEARNING_RATE,
    num_train_epochs=1,
    # fp16=True, # for T4-GPU
    bf16=True, # for A100
    tf32=True, # for A100
    logging_steps=5,
    save_steps=15, # changed from 5
    save_total_limit=2,
    report_to="none",
)

# --- Trainer ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    ),
)

print(" Starting QLoRA repetition fine-tuning...")
trainer.train(resume_from_checkpoint=True)

print(" Saving adapter...")
model.save_pretrained("./final_qlora_adapter")
tokenizer.save_pretrained("./final_qlora_adapter")